# Quick Parquet checks (CRM export)

Set `data_path` to your folder (e.g. `out/pq_0423/`). Requires `pandas`, `pyarrow`.

In [ ]:
import os
from datetime import date, datetime, timezone

import pandas as pd

data_path = "out/pq_0423/"  # trailing slash optional below
data_nm = "client_accounts"


def pq(name: str) -> str:
    root = data_path.rstrip("/\\") + os.sep
    return root + name + ".parquet"


df = pd.read_parquet(pq(data_nm))
df.head()

### 1. Duplicate keys (RM / client line)
- `serving_account_id` should be unique (one row per serving line).
- `(serving_account_id, customer_id)` duplicate = bad if you expect one row per pair.

In [ ]:
# duplicate (serving_account_id, customer_id) — usually empty if PK is serving_account_id
dup_pair = df[df.duplicated(["serving_account_id", "customer_id"], keep=False)]
dup_pair

In [ ]:
dup_sid = df[df.duplicated(["serving_account_id"], keep=False)]
dup_sid  # should be empty

### 2. Active account → `coverage_owner_id` not null; Primary count
Load `coverage_assignments` and join on `serving_account_id`.

In [ ]:
active = df["account_status"].astype(str).str.strip() == "Active"
missing_owner = df[active & df["coverage_owner_id"].isna()]
missing_owner  # should be empty

cov = pd.read_parquet(pq("coverage_assignments"))
prim = cov[cov["assignment_type"].astype(str).str.strip() == "Primary"]
n_prim = prim.groupby(prim["serving_account_id"].astype(str)).size()

act_ids = set(df.loc[active, "serving_account_id"].astype(str))
bad = [(sid, int(n_prim.get(sid, 0))) for sid in sorted(act_ids) if n_prim.get(sid, 0) != 1]
print("Active serving lines without exactly 1 Primary:", bad[:20], "..." if len(bad) > 20 else "")

### 3. `is_servicing_eligible` vs KYC rule
Expected `True` iff: `kyc_status=='Active'`, `kyc_expiry_date` ≥ as-of, `sanctions_status=='Clear'`.
As-of: env `FDS_AS_OF_DATE` or UTC today.

In [ ]:
raw = os.environ.get("FDS_AS_OF_DATE", "").strip()
if raw and len(raw) >= 10:
    y, m, d = (int(x) for x in raw[:10].split("-"))
    as_of = date(y, m, d)
else:
    as_of = datetime.now(timezone.utc).date()
print("as_of:", as_of)

kexp = pd.to_datetime(df["kyc_expiry_date"], utc=True, errors="coerce").dt.date
kyc_ok = df["kyc_status"].astype(str).str.strip() == "Active"
san_ok = df["sanctions_status"].astype(str).str.strip() == "Clear"
exp_ok = kexp >= as_of
expected = kyc_ok & san_ok & exp_ok & kexp.notna()
actual = df["is_servicing_eligible"].astype(bool)
mismatch = df[expected != actual]
mismatch[["serving_account_id", "kyc_status", "kyc_expiry_date", "sanctions_status", "is_servicing_eligible"]]